##### 用不到gee

In [8]:
import ee
ee.Authenticate()
ee.Initialize()

ModuleNotFoundError: No module named 'ee'

In [1]:
import tensorflow as tf
print(tf.__version__)

2.1.0


### 设置GPU显存按需分配

In [3]:
#如果没有装GPU版本，这部分就不用写，但是如果不用GPU加速计算，使用CPU将会严重限制模型的训练速度gpus = tf.config.experimental.list_physical_devices('GPU') 
gpus = tf.config.experimental.list_physical_devices('GPU') 
if gpus:
  try:

    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:

    print(e)

1 Physical GPUs, 1 Logical GPUs


##### 用不到folium

In [2]:
import folium
print(folium.__version__)

ModuleNotFoundError: No module named 'folium'

In [4]:
# Your Earth Engine username.  This is used to import a classified image
# into your Earth Engine assets folder.
USER_NAME = 'ranbingbing'

# Cloud Storage bucket into which training, testing and prediction 
# datasets will be written.  You must be able to write into this bucket.
OUTPUT_BUCKET = 'ice'
REGION = 'us-central1'

# Use these bands for prediction.
#BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2','hue','saturation','value','AWEIsh','ndvi','elevation','slope','Mtpi','Chili','TopographicDiversity','greenness','wetness','brightness']
BANDS = [ 'red', 'nir', 'swir2','value','TopographicDiversity','greenness','wetness','brightness']


# The labels, consecutive integer indices starting from zero, are stored in
# this property, set on each point.
LABEL = 'landcover'
# Number of label values, i.e. number of classes in the classification.
N_CLASSES = 3

# These names are used to specify properties in the export of
# training/testing data and to define the mapping between names and data
# when reading into TensorFlow datasets.
FEATURE_NAMES = list(BANDS)
FEATURE_NAMES.append(LABEL)

# File names for the training and testing datasets.  These TFRecord files
# will be exported from Earth Engine into the Cloud Storage bucket.
TRAIN_FILE_PREFIX = '/trainingsample_trainFile02'
TEST_FILE_PREFIX = 'trainingsample_testFile02'
file_extension = '.tfrecord.gz'
TRAIN_FILE_PATH = 'D:/' + OUTPUT_BUCKET + '/'+ TRAIN_FILE_PREFIX + file_extension
TEST_FILE_PATH = 'D:/' + OUTPUT_BUCKET + '/' + TEST_FILE_PREFIX + file_extension

# File name for the prediction (image) dataset.  The trained model will read
# this dataset and make predictions in each pixel.
IMAGE_FILE_PREFIX = 'image'

# The output path for the classified image (i.e. predictions) TFRecord file.
OUTPUT_IMAGE_FILE = 'D:/' + OUTPUT_BUCKET + '/Classified_pixel_demo02.TFRecord'

# The name of the Earth Engine asset to be created by importing
# the classified image from the TFRecord file in Cloud Storage.
OUTPUT_ASSET_ID = 'users/' + USER_NAME + '/Classified_pixel_demo02'

In [5]:
print('Found training file.' if tf.io.gfile.exists(TRAIN_FILE_PATH) 
    else 'No training file found.')
print('Found testing file.' if tf.io.gfile.exists(TEST_FILE_PATH) 
    else 'No testing file found.')

Found training file.
Found testing file.


In [6]:
# Create a dataset from the TFRecord file in Cloud Storage.
train_dataset = tf.data.TFRecordDataset(TRAIN_FILE_PATH, compression_type='GZIP')
# Print the first record to check.
print(iter(train_dataset).next())

tf.Tensor(b'\n\x89\x02\n\x0f\n\x03red\x12\x08\x12\x06\n\x04\x00\x00+C\n\x12\n\x06random\x12\x08\x12\x06\n\x04\xccfk=\n\x16\n\nbrightness\x12\x08\x12\x06\n\x04\x00\x00\xa0B\n\x15\n\tgreenness\x12\x08\x12\x06\n\x04\x00\x00`\xc2\n\x15\n\tlandcover\x12\x08\x12\x06\n\x04\x00\x00\x80?\n\x0f\n\x03nir\x12\x08\x12\x06\n\x04\x00\x00\x84B\n \n\x14TopographicDiversity\x12\x08\x12\x06\n\x04\x00\x00?C\n\x13\n\x07wetness\x12\x08\x12\x06\n\x04\x00\x00\x10B\n\x11\n\x05swir2\x12\x08\x12\x06\n\x04\x00\x00\xd0A\n\x11\n\x05value\x12\x08\x12\x06\n\x04\x00\x00\xa6B\n.\n\x0csystem:index\x12\x1e\n\x1c\n\x1a1_1_000000000000000005f3_0', shape=(), dtype=string)


In [7]:
# List of fixed-length features, all of which are float32.
columns = [
  tf.io.FixedLenFeature(shape=[1], dtype=tf.float32) for k in FEATURE_NAMES
]

# Dictionary with names as keys, features as values.
features_dict = dict(zip(FEATURE_NAMES, columns))

from pprint import pprint
pprint(features_dict)

{'TopographicDiversity': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'brightness': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'greenness': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'landcover': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'nir': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'red': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'swir2': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'value': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None),
 'wetness': FixedLenFeature(shape=[1], dtype=tf.float32, default_value=None)}


In [8]:
def parse_tfrecord(example_proto):
  """The parsing function.

  Read a serialized example into the structure defined by featuresDict.

  Args:
    example_proto: a serialized Example.

  Returns:
    A tuple of the predictors dictionary and the label, cast to an `int32`.
  """
  parsed_features = tf.io.parse_single_example(example_proto, features_dict)
  labels = parsed_features.pop(LABEL)
  return parsed_features, tf.cast(labels, tf.int32)

# Map the function over the dataset.
parsed_dataset = train_dataset.map(parse_tfrecord, num_parallel_calls=5)

# Print the first parsed record to check.
pprint(iter(parsed_dataset).next())

({'TopographicDiversity': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([191.], dtype=float32)>,
  'brightness': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([80.], dtype=float32)>,
  'greenness': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([-56.], dtype=float32)>,
  'nir': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([66.], dtype=float32)>,
  'red': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([171.], dtype=float32)>,
  'swir2': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([26.], dtype=float32)>,
  'value': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([83.], dtype=float32)>,
  'wetness': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([36.], dtype=float32)>},
 <tf.Tensor: shape=(1,), dtype=int32, numpy=array([1])>)


In [9]:
from tensorflow import keras

input_dataset = parsed_dataset

# Keras requires inputs as a tuple.  Note that the inputs must be in the
# right shape.  Also note that to use the categorical_crossentropy loss,
# the label needs to be turned into a one-hot vector.
def to_tuple(inputs, label):
  return (tf.transpose(list(inputs.values())),
          tf.one_hot(indices=label, depth=N_CLASSES))

# Map the to_tuple function, shuffle and batch.
input_dataset = input_dataset.map(to_tuple).batch(8)

# Define the layers in the model.
model = tf.keras.models.Sequential([
  tf.keras.layers.Dense(64, activation=tf.nn.relu),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(N_CLASSES, activation=tf.nn.softmax)
])

# Compile the model with the specified loss function.
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model to the training data.
model.fit(x=input_dataset, epochs=10)

Epoch 1/10
644/644 [==============================] - 3s 4ms/step - loss: 58.2479 - accuracy: 0.9559
Epoch 2/10
644/644 [==============================] - 2s 2ms/step - loss: 47.1482 - accuracy: 0.9326
Epoch 3/10
644/644 [==============================] - 2s 3ms/step - loss: 41.4646 - accuracy: 0.9288
Epoch 4/10
644/644 [==============================] - 2s 3ms/step - loss: 23.7461 - accuracy: 0.9321
Epoch 5/10
644/644 [==============================] - 2s 3ms/step - loss: 13.7269 - accuracy: 0.9334
Epoch 6/10
644/644 [==============================] - 2s 3ms/step - loss: 6.7014 - accuracy: 0.9173
Epoch 7/10
644/644 [==============================] - 2s 2ms/step - loss: 3.9068 - accuracy: 0.8307
Epoch 8/10
644/644 [==============================] - 1s 2ms/step - loss: 1.6682 - accuracy: 0.7868
Epoch 9/10
644/644 [==============================] - 2s 2ms/step - loss: 1.3694 - accuracy: 0.8024
Epoch 10/10
644/644 [==============================] - 2s 2ms/step - loss: 1.0285 - accuracy: 0

In [10]:
test_dataset = (
  tf.data.TFRecordDataset(TEST_FILE_PATH, compression_type='GZIP')
    .map(parse_tfrecord, num_parallel_calls=5)
    .map(to_tuple)
    .batch(1))

model.evaluate(test_dataset)

   2223/Unknown - 7s 3ms/step - loss: 1.7123 - accuracy: 0.6968

[1.712326472566941, 0.69680613]

### 预测

In [11]:
import os

In [12]:
IMAGE_FILE_PREFIX = 'image'

#files_list = os.listdir('D:/ice/tf04')
o_files_list = os.listdir('D:/ice/tf04')
files_list = []
for f in o_files_list:
    files_list.append('D:/ice/tf04/'+f)

# Get only the files generated by the image export.
exported_files_list = [s for s in files_list if IMAGE_FILE_PREFIX in s]

# Get the list of image files and the JSON mixer file.
image_files_list = []
json_file = None
for f in exported_files_list:
  if f.endswith('.tfrecord.gz'):
    image_files_list.append(f)
  elif f.endswith('.json'):
    json_file = f

# Make sure the files are in the right order.
image_files_list.sort()

pprint(image_files_list)
print(json_file)

['D:/ice/tf04/images00000.tfrecord.gz',
 'D:/ice/tf04/images00001.tfrecord.gz',
 'D:/ice/tf04/images00002.tfrecord.gz',
 'D:/ice/tf04/images00003.tfrecord.gz',
 'D:/ice/tf04/images00004.tfrecord.gz',
 'D:/ice/tf04/images00005.tfrecord.gz',
 'D:/ice/tf04/images00006.tfrecord.gz',
 'D:/ice/tf04/images00007.tfrecord.gz',
 'D:/ice/tf04/images00008.tfrecord.gz',
 'D:/ice/tf04/images00009.tfrecord.gz',
 'D:/ice/tf04/images00010.tfrecord.gz',
 'D:/ice/tf04/images00011.tfrecord.gz',
 'D:/ice/tf04/images00012.tfrecord.gz',
 'D:/ice/tf04/images00013.tfrecord.gz',
 'D:/ice/tf04/images00014.tfrecord.gz',
 'D:/ice/tf04/images00015.tfrecord.gz',
 'D:/ice/tf04/images00016.tfrecord.gz',
 'D:/ice/tf04/images00017.tfrecord.gz',
 'D:/ice/tf04/images00018.tfrecord.gz',
 'D:/ice/tf04/images00019.tfrecord.gz',
 'D:/ice/tf04/images00020.tfrecord.gz',
 'D:/ice/tf04/images00021.tfrecord.gz',
 'D:/ice/tf04/images00022.tfrecord.gz',
 'D:/ice/tf04/images00023.tfrecord.gz',
 'D:/ice/tf04/images00024.tfrecord.gz',


In [25]:
import json

# Load the contents of the mixer file to a JSON object.
json_text = gsutil cat {json_file}
# Get a single string w/ newlines from the IPython.utils.text.SList
mixer = json.loads(json_text.nlstr)
pprint(mixer)

SyntaxError: invalid syntax (<ipython-input-25-562e5337fc6d>, line 4)

##### 上一步运行不出结果，因为我不懂!gsutil cat，所以直接复制了你的结果

In [13]:
mixer = {'patchDimensions': [1024, 1024],
 'patchesPerRow': 611,
 'projection': {'affine': {'doubleMatrix': [0.00026949458523585647,
                                            0.0,
                                            -179.5750219260606,
                                            0.0,
                                            -0.00026949458523585647,
                                            48.037679312876655]},
                'crs': 'EPSG:4326'},
 'totalPatches': 64155}

In [14]:
# Get relevant info from the JSON mixer file.
patch_width = mixer['patchDimensions'][0]
patch_height = mixer['patchDimensions'][1]
patches = mixer['totalPatches']
patch_dimensions_flat = [patch_width * patch_height, 1]

# Note that the tensors are in the shape of a patch, one patch for each band.
image_columns = [
  tf.io.FixedLenFeature(shape=patch_dimensions_flat, dtype=tf.float32) 
    for k in BANDS
]

# Parsing dictionary.
image_features_dict = dict(zip(BANDS, image_columns))

# Note that you can make one dataset from many files by specifying a list.
image_dataset = tf.data.TFRecordDataset(image_files_list, compression_type='GZIP')

# Parsing function.
def parse_image(example_proto):
  return tf.io.parse_single_example(example_proto, image_features_dict)

# Parse the data into tensors, one long tensor per patch.
image_dataset = image_dataset.map(parse_image, num_parallel_calls=5)

# Break our long tensors into many little ones.
image_dataset = image_dataset.flat_map(
  lambda features: tf.data.Dataset.from_tensor_slices(features)
)

# Turn the dictionary in each record into a tuple without a label.
image_dataset = image_dataset.map(
  lambda data_dict: (tf.transpose(list(data_dict.values())), )
)

# Turn each patch into a batch.
image_dataset = image_dataset.batch(patch_width * patch_height)

In [ ]:
# Run prediction in batches, with as many steps as there are patches.
predictions = model.predict(image_dataset, steps=patches, verbose=1)

# Note that the predictions come as a numpy array.  Check the first one.
print(predictions[0])

 1870/64155 [..............................] - ETA: 650:54:09